# B5 — Cancer biology: does predicted damage explain where cancer mutations fall?

**Question.** Are cancer mutations in SH3 domains enriched for predicted destabilisation — by gene,
by tumour site, or by position in the family?

**What it does.** Every group is compared only with random samples of the atlas **matched on
burial composition** (buried / surface / unknown; 2,000 draws), because a more buried domain will
show more predicted damage for trivial reasons. Benjamini–Hochberg across tests. Everything is run
on all variants and on non-hypermutated variants only. The positional test compares the 56
aligned positions with a uniform null; it does **not** model trinucleotide context or codon-level
mutational opportunity.

**Inputs:** B3 calls, A0 atlas, A2 structure. **Runtime:** about 3 min, CPU.
**Outputs:** `gene_enrichment.csv`, `tumour_type_enrichment.csv`, `position_hotspots.csv`, `RESULTS_B5.md`.
Explanation: `docs/B5_cancer_biology.md`.

In [1]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'B5', 'Cancer biology', resume=RESUME)
print(C.env_info())

Mounted at /content/drive
B5 — Cancer biology
  results -> /content/drive/MyDrive/sh3_repro/results/B5_cancer_biology
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'transformers': '5.16.1', 'torch': '2.11.0+cpu', 'gpu': 'none'}


In [2]:
import re
A = pd.read_parquet(R.res('B3', 'atlas_predictions.parquet'))
MASTER = pd.read_csv(R.res('A0', 'SH3_cancer_master.tsv'), sep='\t')
STR = pd.read_csv(R.res('A2', 'structure_features.csv'))
A = A.merge(MASTER[['variant_id', 'n_samples_nonhyper', 'top_primary_sites']].drop_duplicates('variant_id'), on='variant_id', how='left')
A['destab'] = A.fold_lo >= 1.0; A['nonhyper'] = A.n_samples_nonhyper.fillna(0) > 0
A['stratum'] = np.where(A.buried.eq(True), 'buried', np.where(A.buried.eq(False) & A.has_structure.eq(True), 'surface', 'unknown'))

def matched_null(pool, group, n_perm=2000, seed=0):
    rng = np.random.default_rng(seed); ps = pool.stratum.values; pv = pool.destab.values.astype(float)
    idx = {k: np.flatnonzero(ps == k) for k in np.unique(ps)}; need = group.stratum.value_counts()
    if any(len(idx.get(k, ())) < n for k, n in need.items()): return np.nan, np.nan, np.nan
    null = np.empty(n_perm)
    for i in range(n_perm):
        null[i] = sum(pv[rng.choice(idx[k], n, replace=False)].sum() for k, n in need.items()) / need.sum()
    obs = float(group.destab.mean())
    return obs, float(np.median(null)), float((np.sum(null >= obs) + 1) / (n_perm + 1))
def bh(p):
    p = np.asarray(p, float); n = len(p); q = np.empty(n); prev = 1.0
    for rank, i in enumerate(np.argsort(p)[::-1]): prev = min(prev, p[i] * n / (n - rank)); q[i] = prev
    return q
def enrich(pool, key, min_n, n_perm=2000):
    rows = []
    for k, sub in pool.groupby(key):
        if len(sub) < min_n: continue
        o, e, p = matched_null(pool, sub, n_perm)
        if not np.isnan(o): rows.append({key: k, 'n': len(sub), 'frac_destab': o, 'expected': e, 'excess': o - e, 'p': p})
    t = pd.DataFrame(rows); t['q'] = bh(t.p.values); return t.sort_values('p')

In [3]:
t0 = time.time()
GA = enrich(A, 'gene', 20); GN = enrich(A[A.nonhyper], 'gene', 20)
ex = A[['destab', 'stratum']].assign(site=A.top_primary_sites.map(
    lambda v: [re.sub(r'\(.*?\)', '', x).strip().lower() for x in re.split(r'[;,|]', v) if x.strip()]
              if isinstance(v, str) else [])).explode('site').dropna(subset=['site'])
TT = enrich(ex[ex.site != ''], 'site', 30, n_perm=1000)
pd.concat([GA.assign(set='all'), GN.assign(set='nonhyper')]).to_csv(R.out('gene_enrichment.csv'), index=False)
TT.to_csv(R.out('tumour_type_enrichment.csv'), index=False)
print(f'permutation tests {time.time() - t0:.0f}s')
ref = STR[STR.domain_key == 'GRB2-SH3'].set_index('pos')
H = A[A.ref_pos.notna()].groupby('ref_pos').agg(n=('destab', 'size'), n_destab=('destab', 'sum'), mean_fold=('pred_ddG_fold', 'mean'),
                                              mean_bind=('pred_ddG_bind', 'mean'), n_nonhyper=('nonhyper', 'sum')).reset_index()
H['frac_destab'] = H.n_destab / H.n; H['rsasa'] = H.ref_pos.map(ref.rsasa); H['dist_peptide'] = H.ref_pos.map(ref.dist_peptide)
H['buried'] = H.rsasa < 0.25; H['interface'] = H.dist_peptide < 5.0
H.round(4).to_csv(R.out('position_hotspots.csv'), index=False)
obs = H.set_index('ref_pos').n.reindex(range(1, 57), fill_value=0).values; rng = np.random.default_rng(0)
null = np.array([np.bincount(rng.integers(0, 56, obs.sum()), minlength=56).var() for _ in range(5000)])
P_UNI = float((np.sum(null >= obs.var()) + 1) / 5001)
hv = H.dropna(subset=['rsasa'])
R_VD = C.spearman(hv.n, hv.mean_fold); R_DB = C.spearman(hv.mean_fold, -hv.rsasa); R_VB = C.spearman(hv.n, -hv.rsasa)

permutation tests 31s


## Results
The gene table is not printed by name here: one hit at q < 0.1 among 170 tests is what Benjamini–Hochberg allows from noise, and it is not reported as a finding. The full table is in `gene_enrichment.csv`.

In [4]:
for k, v in [('variants', len(A)), ('nonhyper', int(A.nonhyper.sum())), ('frac_destab_all', A.destab.mean()),
             ('frac_destab_nonhyper', A[A.nonhyper].destab.mean()),
             ('genes_tested_all', len(GA)), ('genes_sig_all', int((GA.q < 0.1).sum())),
             ('genes_tested_nonhyper', len(GN)), ('genes_sig_nonhyper', int((GN.q < 0.1).sum())),
             ('sites_tested', len(TT)), ('sites_sig', int((TT.q < 0.1).sum())), ('positions_mapped', len(H)),
             ('positional_uniform_p', P_UNI), ('rho_variants_vs_damage', R_VD), ('rho_damage_vs_burial', R_DB),
             ('rho_variants_vs_burial', R_VB), ('binding_claims_made', False)]:
    R.metric(k, v)
R.section('Enrichment tests', 'Burial-matched permutation null, Benjamini–Hochberg, q < 0.1.', pd.DataFrame([
    dict(test='genes, all variants (≥20 variants)', tested=len(GA), significant=int((GA.q < 0.1).sum()), smallest_q=GA.q.min()),
    dict(test='genes, non-hypermutated only (≥20)', tested=len(GN), significant=int((GN.q < 0.1).sum()), smallest_q=GN.q.min()),
    dict(test='tumour primary sites (≥30)', tested=len(TT), significant=int((TT.q < 0.1).sum()), smallest_q=TT.q.min())]), '{:.3g}')
R.section('Tumour sites', '', TT, '{:.3g}')
R.finish([f"{A.destab.mean():.2%} of atlas variants are called destabilising ({A[A.nonhyper].destab.mean():.2%} of non-hypermutated).",
          f"Genes enriched: {int((GA.q<0.1).sum())}/{len(GA)} (all), {int((GN.q<0.1).sum())}/{len(GN)} (non-hypermutated; "
          f"underpowered); tumour sites {int((TT.q<0.1).sum())}/{len(TT)}.",
          f"Variant counts do not follow predicted damage (ρ {R_VD:+.3f}) while damage follows burial (ρ {R_DB:+.3f}).",
          f"Positions are non-uniform (p = {P_UNI:.1e}), but the null ignores mutational opportunity and the pattern is "
          f"unrelated to damage: a mutational-process observation, not a selection signal."])


B5: REPRODUCED
              quantity  this_run    recorded    status
              variants      6884 6884.000000 identical
              nonhyper      2165 2165.000000 identical
       frac_destab_all  0.059558    0.059558 identical
  frac_destab_nonhyper  0.055889    0.055889 identical
      genes_tested_all       170  170.000000 identical
         genes_sig_all         1    1.000000 identical
 genes_tested_nonhyper        17   17.000000 identical
    genes_sig_nonhyper         0    0.000000 identical
          sites_tested        22   22.000000 identical
             sites_sig         0    0.000000 identical
      positions_mapped        56   56.000000 identical
  positional_uniform_p    0.0002    0.000200 identical
rho_variants_vs_damage -0.008853   -0.008853 identical
  rho_damage_vs_burial  0.711651    0.711651 identical
rho_variants_vs_burial -0.177844   -0.177844 identical

wrote RESULTS_B5.md, metrics_B5.json, comparison_B5.csv


,quantity,this_run,recorded,status
0,variants,6884,6884.000000,identical
1,nonhyper,2165,2165.000000,identical
2,frac_destab_all,0.059558,0.059558,identical
3,frac_destab_nonhyper,0.055889,0.055889,identical
4,genes_tested_all,170,170.000000,identical
5,genes_sig_all,1,1.000000,identical
6,genes_tested_nonhyper,17,17.000000,identical
7,genes_sig_nonhyper,0,0.000000,identical
8,sites_tested,22,22.000000,identical
9,sites_sig,0,0.000000,identical
